# Part 2 — MYH Curated Applications Dataset (2020–2025)

**Notebook role:** This notebook will become the full, rerunnable raw-to-curated workflow for Part 2 of the Data Pipeline Project.

**Current implementation status:**  
Sub-project **2.2** has now implemented the source-file exploration layer: workbook/sheet inventory, header-row detection, `Tabell 3` application-grain evidence, `Tabell 4` grain warning, cross-year schema comparison, and source-value observations that matter for later harmonization.  
The later sections remain structured implementation slots for Sub-projects **2.3–2.7**.

## Project purpose

The finished notebook should:
- read the original MYH Excel workbooks for application rounds **2020–2025**,
- build a longitudinal curated applications dataset from **`Tabell 3`**,
- preserve a clear main-table grain: **one row = one application in one application round**,
- document source differences, harmonization choices, cleaning choices, enrichment, and validation,
- export a final dataset that can later be loaded into SQL and served through a read-oriented API.

## 1. Scope and design principles

### Fixed project direction
- Source years: **2020, 2021, 2022, 2023, 2024, 2025**
- Main source sheet: **`Tabell 3`**
- Main table grain: **one application in one application round per row**
- `Tabell 4` is acknowledged as useful but must **not** be merged into the main applications table in a way that duplicates applications.

### Working quality standard
This notebook should read as an explanatory data journey rather than a code dump.  
Every major transformation will later be accompanied by:
1. what is being done,
2. why it is being done,
3. what trade-off or source inconsistency it addresses.


## 2. Imports and runtime setup

This cell contains the small set of general-purpose libraries expected in the notebook.  
Additional imports should only be added later when they support a concrete implementation need.


In [11]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

## 3. Project paths and folder conventions

The notebook is designed to work whether VS Code runs it from:
- the repository root, or
- the `part_2/` folder itself.

The raw-vs-processed rule is simple:
- `data/raw/` contains the unchanged MYH Excel inputs,
- `data/processed/` contains notebook-generated exports.


In [12]:
def resolve_part_2_dir() -> Path:
    """Return the Part 2 workspace when run from repo root or from part_2/."""
    cwd = Path.cwd().resolve()

    if cwd.name == "part_2":
        return cwd

    candidate = cwd / "part_2"
    if candidate.exists():
        return candidate

    raise FileNotFoundError(
        "Could not locate the 'part_2' folder. "
        "Run this notebook from the repository root or from the part_2/ folder."
    )


PART_2_DIR = resolve_part_2_dir()
RAW_DATA_DIR = PART_2_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PART_2_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Part 2 workspace: {PART_2_DIR}")
print(f"Raw input folder: {RAW_DATA_DIR}")
print(f"Processed export folder: {PROCESSED_DATA_DIR}")


Part 2 workspace: C:\Users\Skolkonto\source\repos\VSCode\de25-hemtenta-datapipelinesteam\part_2
Raw input folder: C:\Users\Skolkonto\source\repos\VSCode\de25-hemtenta-datapipelinesteam\part_2\data\raw
Processed export folder: C:\Users\Skolkonto\source\repos\VSCode\de25-hemtenta-datapipelinesteam\part_2\data\processed


## 4. Raw-data inventory check

This small setup check confirms which Excel files are currently present in `data/raw/`.  
The full workbook and sheet exploration belongs to **Sub-project 2.2**, but this inventory makes the workspace immediately usable.


In [13]:
EXPECTED_SOURCE_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
SOURCE_YEAR_RE = re.compile(r"(20\d{2})")

raw_excel_files = sorted(RAW_DATA_DIR.glob("*.xlsx"))

print(f"Excel workbooks currently found: {len(raw_excel_files)}")
for file_path in raw_excel_files:
    print(f"- {file_path.name}")

if not raw_excel_files:
    print(
        "\nNo raw Excel files have been found yet. "
        "Place the six original MYH 2020–2025 workbooks in data/raw/ "
        "before starting the source exploration section."
    )

Excel workbooks currently found: 6
- resultat-ansokningsomgang-2020.xlsx
- resultat-ansokningsomgang-2021.xlsx
- resultat-ansokningsomgang-2022.xlsx
- resultat-ansokningsomgang-2023.xlsx
- resultat-ansokningsomgang-2024.xlsx
- resultat-ansokningsomgang-2025.xlsx


## 5. Source-file understanding

Before designing the curated target table, the notebook first profiles the source workbooks directly.  
This section is deliberately **exploratory but rerunnable**: it turns important source assumptions into visible evidence rather than relying on informal notes.

The profiling below answers six practical questions:
1. Which workbooks and sheets are present?
2. Where are the real headers located in the relevant source tables?
3. Does `Tabell 3` support the intended one-application-per-row main table?
4. Why must `Tabell 4` not be blindly merged into that main table?
5. How much does the `Tabell 3` schema change between 2020 and 2025?
6. Which source-value differences already foreshadow later harmonization work?

### 5.1 Workbook and sheet inventory

The source set is expected to contain one MYH workbook for each application round from **2020** through **2025**.  
This inventory checks both the file set and the internal worksheet layout.

In [14]:
def extract_source_year(file_path: Path) -> int:
    """Extract the MYH application-round year from a workbook filename."""
    match = SOURCE_YEAR_RE.search(file_path.name)
    if match is None:
        raise ValueError(f"Could not extract a source year from: {file_path.name}")
    return int(match.group(1))

source_workbooks = [
    {
        "source_year": extract_source_year(file_path),
        "source_file": file_path.name,
        "file_path": file_path,
    }
    for file_path in raw_excel_files
]

observed_source_years = sorted(workbook["source_year"] for workbook in source_workbooks)
missing_source_years = sorted(set(EXPECTED_SOURCE_YEARS) - set(observed_source_years))
unexpected_source_years = sorted(set(observed_source_years) - set(EXPECTED_SOURCE_YEARS))

if missing_source_years:
    raise FileNotFoundError(
        "Missing expected MYH source workbook(s) for year(s): "
        f"{missing_source_years}."
    )

if unexpected_source_years:
    print(
        "Note: additional Excel workbook years were found outside the planned 2020–2025 scope: "
        f"{unexpected_source_years}. They are visible in the inventory but are not part of the agreed Part 2 scope."
    )

workbook_inventory_rows = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    excel_file = pd.ExcelFile(workbook["file_path"])
    sheet_names = excel_file.sheet_names
    definition_sheet_name = next(
        (sheet for sheet in sheet_names if sheet.startswith("Definitioner")),
        None,
    )
    workbook_inventory_rows.append(
        {
            "source_year": workbook["source_year"],
            "source_file": workbook["source_file"],
            "sheet_count": len(sheet_names),
            "definition_sheet": definition_sheet_name,
            "sheet_names": " | ".join(sheet_names),
        }
    )

workbook_inventory = pd.DataFrame(workbook_inventory_rows).sort_values("source_year").reset_index(drop=True)

display(workbook_inventory)

,source_year,source_file,sheet_count,definition_sheet,sheet_names
0,2020,resultat-ansokningsomgang-2020.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
1,2021,resultat-ansokningsomgang-2021.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
2,2022,resultat-ansokningsomgang-2022.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
3,2023,resultat-ansokningsomgang-2023.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
4,2024,resultat-ansokningsomgang-2024.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...
5,2025,resultat-ansokningsomgang-2025.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...


### 5.2 Header-row detection for `Tabell 3` and `Tabell 4`

The relevant tables do **not** start on the same Excel row in every year.  
Instead of hard-coding the result as prose only, this profiling step scans the first rows of each table and locates the row containing `Diarienummer`, which is part of the true header row.

The output records both:
- the human-facing Excel row number, and
- the zero-based `header=` index that `pandas.read_excel()` would use later.

In [15]:
PROFILE_SHEETS = ["Tabell 3", "Tabell 4"]
HEADER_SCAN_ROWS = 20
HEADER_MARKER = "Diarienummer"


def detect_header_row(file_path: Path, sheet_name: str, marker: str = HEADER_MARKER) -> int:
    """Return the zero-based row index containing the table header marker."""
    preview = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        nrows=HEADER_SCAN_ROWS,
    )

    matching_rows = []
    for row_index in range(len(preview)):
        row_values = {
            str(value).strip()
            for value in preview.iloc[row_index].tolist()
            if pd.notna(value)
        }
        if marker in row_values:
            matching_rows.append(row_index)

    if not matching_rows:
        raise ValueError(
            f"Could not find header marker '{marker}' in the first {HEADER_SCAN_ROWS} rows "
            f"of sheet '{sheet_name}' in '{file_path.name}'."
        )

    return matching_rows[0]


header_row_records = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    for sheet_name in PROFILE_SHEETS:
        header_index = detect_header_row(workbook["file_path"], sheet_name)
        header_row_records.append(
            {
                "source_year": workbook["source_year"],
                "source_sheet": sheet_name,
                "excel_header_row": header_index + 1,
                "pandas_header_index": header_index,
            }
        )

header_row_summary = pd.DataFrame(header_row_records).sort_values(
    ["source_sheet", "source_year"]
).reset_index(drop=True)

header_index_lookup = {
    (row.source_year, row.source_sheet): int(row.pandas_header_index)
    for row in header_row_summary.itertuples(index=False)
}

display(header_row_summary)

,source_year,source_sheet,excel_header_row,pandas_header_index
0,2020,Tabell 3,1,0
1,2021,Tabell 3,1,0
2,2022,Tabell 3,1,0
3,2023,Tabell 3,6,5
4,2024,Tabell 3,6,5
5,2025,Tabell 3,7,6
6,2020,Tabell 4,1,0
7,2021,Tabell 4,1,0
8,2022,Tabell 4,1,0
9,2023,Tabell 4,6,5


### 5.3 Read cleaned profiling copies of the source tables

The helper below is **not yet the production ingestion pipeline**.  
Its role in Sub-project 2.2 is narrower: read each profiled source table consistently enough to count rows, inspect identifiers, and compare source schemas.

In [16]:
def read_profile_table(file_path: Path, sheet_name: str, header_index: int) -> pd.DataFrame:
    """Read a source sheet for profiling and remove purely empty rows/columns."""
    table = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_index,
    )
    table = table.dropna(how="all").dropna(axis=1, how="all")
    table.columns = [str(column).strip() for column in table.columns]
    return table


tabell_3_tables = {}
tabell_4_tables = {}
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    year = workbook["source_year"]
    file_path = workbook["file_path"]
    tabell_3_tables[year] = read_profile_table(
        file_path,
        "Tabell 3",
        header_index_lookup[(year, "Tabell 3")],
    )
    tabell_4_tables[year] = read_profile_table(
        file_path,
        "Tabell 4",
        header_index_lookup[(year, "Tabell 4")],
    )

print(f"Profiling copies created for Tabell 3: {sorted(tabell_3_tables)}")
print(f"Profiling copies created for Tabell 4: {sorted(tabell_4_tables)}")

Profiling copies created for Tabell 3: [2020, 2021, 2022, 2023, 2024, 2025]
Profiling copies created for Tabell 4: [2020, 2021, 2022, 2023, 2024, 2025]


### 5.4 `Tabell 3` profile: evidence for the main applications table

The intended main table grain is:
> **one row = one application in one application round**.

For that to be credible, `Tabell 3` should have:
- a clear row count by year,
- no missing `Diarienummer`, and
- no duplicate `Diarienummer` within a year.

In [17]:
tabell_3_profile_rows = []
for year, table in sorted(tabell_3_tables.items()):
    identifiers = table["Diarienummer"]
    tabell_3_profile_rows.append(
        {
            "source_year": year,
            "tabell_3_rows": len(table),
            "tabell_3_columns": len(table.columns),
            "missing_diarienummer": int(identifiers.isna().sum()),
            "duplicate_diarienummer": int(identifiers.duplicated().sum()),
        }
    )

tabell_3_profile = pd.DataFrame(tabell_3_profile_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_3_profile)

assert int(tabell_3_profile["missing_diarienummer"].sum()) == 0, "Unexpected missing Diarienummer in Tabell 3."
assert int(tabell_3_profile["duplicate_diarienummer"].sum()) == 0, "Unexpected duplicate Diarienummer in Tabell 3."

,source_year,tabell_3_rows,tabell_3_columns,missing_diarienummer,duplicate_diarienummer
0,2020,1482,16,0,0
1,2021,1238,17,0,0
2,2022,1207,16,0,0
3,2023,1258,28,0,0
4,2024,1272,28,0,0
5,2025,1184,28,0,0


### 5.5 `Tabell 4` profile: grain warning before any future joins

`Tabell 4` is useful, but it is **not** at the same grain as the planned main applications table.  
The check below compares total rows with the number of unique `Diarienummer` values.  A positive difference shows that at least some applications appear multiple times in `Tabell 4`.

In [18]:
tabell_4_grain_rows = []
for year, table in sorted(tabell_4_tables.items()):
    unique_application_ids = int(table["Diarienummer"].nunique(dropna=True))
    tabell_4_grain_rows.append(
        {
            "source_year": year,
            "tabell_4_rows": len(table),
            "unique_diarienummer": unique_application_ids,
            "rows_above_unique_application_count": len(table) - unique_application_ids,
        }
    )

tabell_4_grain_check = pd.DataFrame(tabell_4_grain_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_4_grain_check)

assert (
    tabell_4_grain_check["rows_above_unique_application_count"] > 0
).all(), "Expected Tabell 4 to show repeated application identifiers in every profiled year."

,source_year,tabell_4_rows,unique_diarienummer,rows_above_unique_application_count
0,2020,1903,1482,421
1,2021,1586,1238,348
2,2022,1621,1207,414
3,2023,642,243,399
4,2024,842,292,550
5,2025,836,295,541


### 5.6 `Tabell 3` schema comparison across 2020–2025

A cross-year curated dataset cannot assume that every source year has the same input structure.  
This comparison records:
- the column count by year, and
- which source columns appear in which years.

The result gives concrete evidence for the later harmonization/specification work in Sub-project 2.3.

In [19]:
tabell_3_schema_by_year = {
    year: list(table.columns)
    for year, table in sorted(tabell_3_tables.items())
}

schema_count_summary = pd.DataFrame(
    [
        {
            "source_year": year,
            "tabell_3_columns": len(columns),
            "column_names": " | ".join(columns),
        }
        for year, columns in tabell_3_schema_by_year.items()
    ]
).sort_values("source_year").reset_index(drop=True)

all_tabell_3_columns = sorted(
    set().union(*(set(columns) for columns in tabell_3_schema_by_year.values()))
)

schema_presence_rows = []
for column_name in all_tabell_3_columns:
    years_present = [
        year
        for year, columns in tabell_3_schema_by_year.items()
        if column_name in columns
    ]
    schema_presence_rows.append(
        {
            "column_name": column_name,
            "years_present": ", ".join(str(year) for year in years_present),
            "present_in_year_count": len(years_present),
        }
    )

schema_presence_summary = pd.DataFrame(schema_presence_rows).sort_values(
    ["present_in_year_count", "column_name"],
    ascending=[False, True],
).reset_index(drop=True)

schema_variation_summary = schema_presence_summary[
    schema_presence_summary["present_in_year_count"] < len(EXPECTED_SOURCE_YEARS)
].reset_index(drop=True)


with pd.option_context("display.max_colwidth", None):
    display(schema_count_summary)
    
display(schema_variation_summary)

,source_year,tabell_3_columns,column_names
0,2020,16,Utbildningsområde | Utbildningsnamn | Län | Kommun | Flera studiekommuner | Antal kommuner | Antal län | YH-poäng | Studieform | Studietakt % | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar | Diarienummer | Beslut
1,2021,17,Utbildningsområde | Utbildningsnamn | Län | Kommun | Flera studiekommuner | Antal kommuner | Antal län | YH-poäng | Studieform | Studietakt % | Typ av examen | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar | Diarienummer | Beslut
2,2022,16,Utbildningsområde | Utbildningsnamn | Beslut | Diarienummer | Län | Kommun | Flera kommuner | Antal kommuner | YH-poäng | Studieform | Studietakt % | Typ av examen | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar
3,2023,28,Utbildningsområde | SUN5 inriktning | SUN5 inriktning namn | Utbildningsnamn | Beslut | Diarienummer | Flera kommuner | Antal kommuner | Län | Kommun | YH-poäng | Studieform | Studietakt % | Typ av examen | SeQF nivå | Smalt yrkesområde | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar | Sökta platser per utbildningsomgång | Sökta platser totalt | Beviljade platser utbildningsomgång 1 | Beviljade platser utbildningsomgång 2 | Beviljade platser utbildningsomgång 3 | Beviljade platser utbildningsomgång 4 | Beviljade platser utbildningsomgång 5 | Beviljade platser totalt
4,2024,28,Utbildningsområde | SUN5 inriktning | SUN5 inriktning namn | Utbildningsnamn | Beslut | Diarienummer | Flera kommuner | Antal kommuner | Län | Kommun | YH-poäng | Studieform | Studietakt % | Typ av examen | SeQF nivå | Smalt yrkesområde | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar | Sökta platser per utbildningsomgång | Sökta platser totalt | Beviljade platser utbildningsomgång 1 | Beviljade platser utbildningsomgång 2 | Beviljade platser utbildningsomgång 3 | Beviljade platser utbildningsomgång 4 | Beviljade platser utbildningsomgång 5 | Beviljade platser totalt
5,2025,28,Utbildningsområde | SUN5 inriktning | SUN5 inriktning namn | Utbildningsnamn | Beslut | Diarienummer | Flera kommuner | Antal kommuner | Län | Kommun | YH-poäng | Studieform | Studietakt % | Examenstyp | SeQF nivå | Smalt yrkesområde | Utbildningsanordnare administrativ enhet | Huvudmannatyp | Sökta utbildningsomgångar | Beviljade utbildningsomgångar | Sökta platser per utbildningsomgång | Sökta platser totalt | Beviljade platser utbildningsomgång 1 | Beviljade platser utbildningsomgång 2 | Beviljade platser utbildningsomgång 3 | Beviljade platser utbildningsomgång 4 | Beviljade platser utbildningsomgång 5 | Beviljade platser totalt


,column_name,years_present,present_in_year_count
0,Flera kommuner,"2022, 2023, 2024, 2025",4
1,Typ av examen,"2021, 2022, 2023, 2024",4
2,Beviljade platser totalt,"2023, 2024, 2025",3
3,Beviljade platser utbildningsomgång 1,"2023, 2024, 2025",3
4,Beviljade platser utbildningsomgång 2,"2023, 2024, 2025",3
5,Beviljade platser utbildningsomgång 3,"2023, 2024, 2025",3
6,Beviljade platser utbildningsomgång 4,"2023, 2024, 2025",3
7,Beviljade platser utbildningsomgång 5,"2023, 2024, 2025",3
8,SUN5 inriktning,"2023, 2024, 2025",3
9,SUN5 inriktning namn,"2023, 2024, 2025",3


### 5.7 Source values that already signal later harmonization needs

This is still source exploration, not cleaning.  
However, two columns already show year-to-year vocabulary differences that will matter later:
- `Beslut`, where rejection wording changes and 2025 adds `Återkallad`,
- `Huvudmannatyp`, where `Landsting` is replaced by `Region` in later years.

In [20]:
def summarize_distinct_values(tables_by_year: dict[int, pd.DataFrame], column_name: str) -> pd.DataFrame:
    """Summarize sorted distinct non-null source values for one column by source year."""
    rows = []
    for year, table in sorted(tables_by_year.items()):
        distinct_values = sorted(str(value) for value in table[column_name].dropna().unique())
        rows.append(
            {
                "source_year": year,
                "source_column": column_name,
                "distinct_values": " | ".join(distinct_values),
            }
        )
    return pd.DataFrame(rows)

beslut_value_summary = summarize_distinct_values(tabell_3_tables, "Beslut")
huvudmannatyp_value_summary = summarize_distinct_values(tabell_3_tables, "Huvudmannatyp")

display(beslut_value_summary)
display(huvudmannatyp_value_summary)

,source_year,source_column,distinct_values
0,2020,Beslut,Beviljad | Ej beviljad
1,2021,Beslut,Beviljad | Ej beviljad
2,2022,Beslut,Avslag | Beviljad
3,2023,Beslut,Avslag | Beviljad
4,2024,Beslut,Avslag | Beviljad
5,2025,Beslut,Avslag | Beviljad | Återkallad


,source_year,source_column,distinct_values
0,2020,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
1,2021,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
2,2022,Huvudmannatyp,Kommun | Privat | Region | Statlig
3,2023,Huvudmannatyp,Kommun | Privat | Region | Statlig
4,2024,Huvudmannatyp,Kommun | Privat | Region | Statlig
5,2025,Huvudmannatyp,Kommun | Privat | Region


## 6. Why `Tabell 3` is the main source and `Tabell 4` is not merged

The profiling evidence above supports the project’s current data-grain decision:

1. **`Tabell 3` fits the planned main table grain.**  
   Across all six source years, `Tabell 3` has one row per recorded application, zero missing `Diarienummer`, and zero duplicate `Diarienummer` values within each year.

2. **`Tabell 4` is at a more detailed grain.**  
   In every year, `Tabell 4` has more rows than unique `Diarienummer` values. That means some applications appear multiple times there.

3. **Therefore, `Tabell 4` must not be blindly joined into the main applications table.**  
   A direct merge would risk duplicating applications and corrupting counts. If `Tabell 4` is ever used later, it should be handled as a separate more-detailed table or joined only after an explicit aggregation/design decision.

This is a **grain conclusion**, not a final schema conclusion. The formal target schema and source-to-target harmonization rules belong to Sub-project **2.3**.

## 7. Initial curated-table design outline *(to be finalized in Sub-project 2.3)*

The final curated schema is not locked in Sub-project 2.1.  
However, the project already has a clear design direction:

### Required characteristics
- stable, SQL/API-friendly field names,
- one row per application per application round,
- traceability fields such as `source_year`, `source_file`, `source_sheet`, and preferably `source_row`,
- raw source values retained where helpful,
- normalized companion fields added where cross-year comparison requires harmonization.

### Examples of already anticipated normalization
- source decision values → `beslut_normalized`,
- source provider-type values → `huvudmannatyp_normalized`.

The formal source-to-target mapping table and final column order will be settled later.


## 8. Reusable ingestion and standardization *(Sub-project 2.4)*

This section will later contain:
- explicit file metadata/configuration,
- year-specific `header=` handling for `Tabell 3`,
- reusable reading logic,
- harmonized column names,
- concatenation into a combined longitudinal base table.


## 9. Cleaning, normalization, and enrichment *(Sub-project 2.5)*

This section will later contain:
- datatype conversions,
- safe text cleanup,
- normalized decision categories,
- normalized provider-type categories,
- selected useful derived fields,
- checks confirming the transformations are safe and complete.


## 10. Validation and quality checks *(Sub-project 2.6)*

This section will later implement stronger-than-minimal confidence checks, such as:
- row counts by source year,
- missing identifier checks,
- duplicate `diarienummer` checks within each year,
- normalized-value mapping coverage,
- missingness review,
- datatype checks,
- sanity summaries by year and category.


## 11. Export of the curated dataset *(Sub-project 2.6)*

The final curated dataset will be written into:

```text
part_2/data/processed/
```

CSV is the expected baseline export.  
Whether a Parquet export adds enough value to keep will be decided later, not prematurely.


## 12. SQL/API handoff note and final reflection *(Sub-project 2.7)*

This final section will explain:
- what table was exported,
- its grain and intended downstream use,
- how it supports later SQL loading,
- how it can support simple read-oriented FastAPI endpoints,
- what limitations or future extensions remain.


## Sub-project 2.2 checkpoint

The source-exploration phase is complete when:
- this notebook contains a rerunnable workbook and sheet inventory,
- the real header-row positions for `Tabell 3` and `Tabell 4` are demonstrated,
- `Tabell 3` is profiled with row counts, column counts, and identifier-quality checks,
- `Tabell 4` is shown to have a more detailed grain than the planned main applications table,
- the cross-year `Tabell 3` schema variation is made visible,
- source-value changes in `Beslut` and `Huvudmannatyp` are surfaced for later harmonization,
- the handoff points cleanly into Sub-project **2.3 — Target schema and harmonization specification**.